In [ ]:
import warnings
warnings.filterwarnings("ignore")

from monai.data import Dataset, DataLoader
from monai.utils import set_determinism

TensorRT-LLM is not installed. Please install TensorRT-LLM or set TRTLLM_PLUGINS_PATH to the directory containing libnvinfer_plugin_tensorrt_llm.so to use converters for torch.distributed ops


[10/21/2025-11:24:11] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


In [ ]:
import sys
import os

parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

from src.data.temporal_loader import load_temporal_splits_from_json
from src.data.transforms import build_pretrain_transform

from src.data.utils import images_only_dataset
from src.networks.vitautoenc import ViTAutoEnc
from src.train.pretrain_ops import pretrain

In [ ]:
img_size = (128, 128, 64)

# Create transform
transform = build_pretrain_transform()

In [4]:
# Set seed
seed = 42
set_determinism(seed)

# Assemble per-patient sequences
root = "/standard/gam_ai_group/new T1_split"
splits = load_temporal_splits_from_json(root)
X_train, X_val, X_test = splits["train"], splits["val"], splits["test"]

In [ ]:
# Create MONAI dataset
train_image_data = images_only_dataset(X_train)
val_image_data = images_only_dataset(X_val)
train_dataset = Dataset(data=train_image_data, transform=transform)
val_dataset = Dataset(data=val_image_data, transform=transform)

In [ ]:
batch_size = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
)

# TemporalUNETR

Building and testing the TemporalUNETR

In [ ]:
net = ViTAutoEnc(
    in_channels=1,
    img_size=img_size,
    patch_size=16,
    dropout_rate=0.1
)

In [ ]:
net, some_output = pretrain(
    net,
    train_loader,
    val_loader,
    max_epochs=30,
    batch_size=batch_size
)

2025-10-21 12:09:58,802 - INFO - Using device: cuda
2025-10-21 12:09:58,803 - INFO - Training configuration - Epochs: 30, Initial LR: 0.0001, Val interval: 1
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2025-10-21 12:09:58,898 - INFO - Initialized Accelerator with standard precision
2025-10-21 12:09:59,231 - INFO - Initialized loss functions (tverskyce), optimizer (AdamW), and scheduler (onecycle))
2025-10-21 12:09:59,275 - INFO - Initialized 6 validation metrics: ['dice mean', 'rvd mean', 'hausdorff mean', 'surface dice mean', 'precision mean', 'recall mean']
2025-10-21 12:09:59,275 - INFO - Starting epoch 1/30
2025-10-21 12:10:38,200 - INFO - Epoch 1 - Batch 1/29 - Train loss: 1.5131


In [ ]:
path = "test_save.pth"
net.save_vit_weights(path)